### Understanding how input - target helps llm's in prediciton
- This step is applied in data pre-training when BPE encodes raw data in token id's
- Data's are segregated in input tokens and target tokens
- Stride, context window and batch sizes parameters are used to convert data into input and target tokens pairs

In [1]:
!pip install tiktoken

In [12]:
import tiktoken
# Understanding core principle behind input - target pair
text1 = "My Name is Priyanshu Mishra. I want to be a life-long learner."
text2 = "This means you need to work hard and do continuous learning"

raw_text = "<|endoftext|>".join([text1, text2])

#print(raw_text)

# Tokenized the raw_text using BPE (Byte Pair Encoding) algorithm (sub-word algorithm)
tokenizer = tiktoken.get_encoding("gpt2")

#Encoding the text
tokens = tokenizer.encode(raw_text, allowed_special= {"<|endoftext|>"})

print(tokens)


[3666, 6530, 318, 4389, 88, 504, 13415, 39523, 430, 13, 314, 765, 284, 307, 257, 1204, 12, 6511, 22454, 1008, 13, 50256, 1212, 1724, 345, 761, 284, 670, 1327, 290, 466, 12948, 4673]


In [3]:
# Sample text to make use of input ids and target ids batches
context_window = 5
input_ids = []
target_ids = []
for i in range(1, context_window + 1):
    input_ids = tokens[:i]
    target_ids = tokens[i]

    print(f'input_ids: {input_ids} ----> target_ids: {target_ids}')
    #Decode the encoded tokens
    print(f'input_decoded: {tokenizer.decode(input_ids)}  ---> target_decoded: {tokenizer.decode([target_ids])}')
    print('\n')



input_ids: [3666] ----> target_ids: 6530
input_decoded: My  ---> target_decoded:  Name


input_ids: [3666, 6530] ----> target_ids: 318
input_decoded: My Name  ---> target_decoded:  is


input_ids: [3666, 6530, 318] ----> target_ids: 4389
input_decoded: My Name is  ---> target_decoded:  Pri


input_ids: [3666, 6530, 318, 4389] ----> target_ids: 88
input_decoded: My Name is Pri  ---> target_decoded: y


input_ids: [3666, 6530, 318, 4389, 88] ----> target_ids: 504
input_decoded: My Name is Priy  ---> target_decoded: ans




### Datasets and Dataloader in pytorch
1. Datasets: - This modular abstraction block helps to pre-trained datasets on batched size of input_ids and target_ids
2. Dataloader: - Module block help us to train and run datasets defined in the block.

In [20]:
# Understand how context window & batch size works for input_ids & target_ids
context_window = 1
batch_size = 4
input_ids, target_ids
for i in range(0, len(tokens) - batch_size, context_window):
    input_ids = tokens[i: i + batch_size]
    target_ids = tokens[i + 1: i + batch_size + 1]

    print(f'input_ids: {input_ids} ----> target_ids: {target_ids}')
    #Decode the encoded tokens
    print(f'input_decoded: {tokenizer.decode(input_ids)}  ---> target_decoded: {tokenizer.decode(target_ids)}')
    print('\n')



input_ids: [3666, 6530, 318, 4389] ----> target_ids: [6530, 318, 4389, 88]
input_decoded: My Name is Pri  ---> target_decoded:  Name is Priy


input_ids: [6530, 318, 4389, 88] ----> target_ids: [318, 4389, 88, 504]
input_decoded:  Name is Priy  ---> target_decoded:  is Priyans


input_ids: [318, 4389, 88, 504] ----> target_ids: [4389, 88, 504, 13415]
input_decoded:  is Priyans  ---> target_decoded:  Priyanshu


input_ids: [4389, 88, 504, 13415] ----> target_ids: [88, 504, 13415, 39523]
input_decoded:  Priyanshu  ---> target_decoded: yanshu Mish


input_ids: [88, 504, 13415, 39523] ----> target_ids: [504, 13415, 39523, 430]
input_decoded: yanshu Mish  ---> target_decoded: anshu Mishra


input_ids: [504, 13415, 39523, 430] ----> target_ids: [13415, 39523, 430, 13]
input_decoded: anshu Mishra  ---> target_decoded: hu Mishra.


input_ids: [13415, 39523, 430, 13] ----> target_ids: [39523, 430, 13, 314]
input_decoded: hu Mishra.  ---> target_decoded:  Mishra. I


input_ids: [39523, 430, 13, 

In [5]:
import torch

print("Pytorch version: ", torch.__version__)

Pytorch version:  2.10.0+cpu


In [42]:
# Encoding "The verdict text file and converting into token_ids"
with open("the-verdict.txt", "r", encoding= "utf-8") as f:
    raw_data = f.read()


In [46]:
from torch.utils.data import Dataset, DataLoader

# Datasets
class GPTDatasetsV1(Dataset):
    def __init__(self, text, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Encode all the text into token ids
        token_ids = tokenizer.encode(text, allowed_special= {"<|endoftext|>"})

        #Converting token id into batches of input_ids and target_ids
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i: i + max_length]
            target_chunk = token_ids[i + 1: i + 1 + max_length]

            #Converting these batch chunks to input_ids & target_ids tensors
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))


    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

    def __len__(self):
        return len(self.input_ids)


In [61]:
# Creating dataloader to run the datasets in input_ids and target_ids batches
def create_dataloader_v1(txt, batch_size= 5, max_length= 5, stride= 2, num_workers= 0, shuffle= True, drop_last= True):

    #Tokenization of data
    tokenizer = tiktoken.get_encoding("gpt2")

    #Datasets
    datasets = GPTDatasetsV1(txt, tokenizer, max_length, stride)

    #DataLoader
    dataloader = DataLoader(
        datasets,
        batch_size= batch_size,
        shuffle= shuffle,
        num_workers= num_workers,
        drop_last= drop_last
    )

    return dataloader

# Getting dataloader values
data = create_dataloader_v1(raw_data)

In [62]:
first_data = iter(data)
next(first_data)
#next(first_data)

[tensor([[  340,    11,   314,   815,   423],
         [  714,   423,  1813,  4544,  9325],
         [  743,   307, 41746, 12004,   262],
         [  262,   705,  4976,     6,   582],
         [  351,   326,  1808,   319,   340]]),
 tensor([[   11,   314,   815,   423,  1760],
         [  423,  1813,  4544,  9325,   701],
         [  307, 41746, 12004,   262,  6473],
         [  705,  4976,     6,   582,    11],
         [  326,  1808,   319,   340,    11]])]